# 🏴‍☠️ AN2DL Challenge 1: Pirate Pain Dataset

## 🪄 Notebook 02: Feature Engineering

This notebook focuses on feature engineering techniques applied to the Pirate Pain Dataset.

### 😴 TL;DR

To improve model performance on the Pirate Pain Dataset, we enhance the dynamic numerical features by computing temporal derivatives $\Delta$ and rolling standard deviations $\sigma$ for each joint movement feature. This results in a richer representation of joint dynamics, increasing the number of dynamic numerical features from 30 to 90, which helps the model better capture motion and instability patterns.

---

### 🤔 Problem Context

There are 30 dynamic numerical features representing joint movements that form continuous time series over 160 timesteps. Each feature can be thought of as a sensor reading that evolves over time. The main problem arises when we feed the model only raw values directly. For example, consider the following joint data:

```
joint_03(t=0..159) = [0.91, 0.93, 0.97, 0.95, …]
```

The model must discover important temporal patterns on its own. For example, an increase in joint values may indicate worsening pain. Highly variable joint values may indicate instability, while flat joint values may indicate stability.

This could be great for large datasets because the model can implicitly learn these patterns over time. However, since we have a very small dataset of only 661 samples, our model may have difficulty extracting derivatives and local noise levels reliably.

---

### ✅ Solution: Precompute Temporal Derivatives and Rolling Statistics

We can assist the model by providing hints, much like a teacher would. Rather than simply providing raw joint values, we can compute additional features that capture the joints' dynamics over time.

The two main features we can derive are:

1. **Temporal Derivative** (Delta $\Delta$): captures **how much each joint changes from one timestep to the next**. It gives the model a sense of speed and direction of movement (trend/speed).
2. **Rolling Standard Deviation** (Sigma $\sigma$): captures **how much the joint values fluctuate over a short window** (e.g., last 5 timesteps). It gives the model a sense of volatility or stability.

Together, these features provide a richer representation of the joint dynamics, because they explicitly encode _motion_ and _instability_.

- _Motion_: how fast and in what direction the joint is moving (increasing/decreasing).
- _Instability_: how erratic or stable the joint's movement is (high/low variance).

---

### 🧐 Expected effects on Model Performance

The dataset provides 30 raw joints ($x$) that capture absolute positions, which is useful for baseline behavior. Adding the delta ($\Delta$) helps the model detect velocity and direction changes, which may indicate transitions in pain levels (e.g., pain increasing or decreasing). Finally, the rolling standard deviation ($\sigma$) helps the model recognize irregular patterns that may correspond to unstable joint movements, such as sudden jerks or tremors. After all, we are talking about pirates, who may have erratic movements!

So, the new feature set for each joint becomes:
```
D_dyn_num = [joint_01_raw, joint_01_delta, joint_01_sigma,
              joint_02_raw, joint_02_delta, joint_02_sigma,
              ...
              joint_30_raw, joint_30_delta, joint_30_sigma]
```

And the new tensor shape becomes:
```
D_dyn_num = 30 raw + 30 delta + 30 rolling_std = 90 channels
```

That is a **3x increase** in the number of dynamic numerical features, but it is a small price to pay for the richer temporal representation.

What we expect to see, is that the model will now be able to spot difficult patterns like:
> 🧙‍♂️ **Model**: _"Pain increases when joint_03 rapidly increases (high delta) and becomes unstable (high sigma), this indicates worsening pain!"_
>
> 🤓 **Me**: _"Wow, you're either learning or overfitting!"_

This is free because we aren't changing the model architecture, we're only adjusting the inputs.


### 🛠️ Implementation

#### $\Delta$ Compute Delta function

The `compute_delta` function calculates the temporal derivative (delta) of the input tensor `x` along the time axis. The first timestep's delta is set to zero since there is no previous value to compare against. For subsequent timesteps, the delta is computed as the difference between the current and previous timestep values.

---

#### $\sigma$ Rolling Standard Deviation function

About the `rolling_std` function, there is no tricks, no gimmicks. It uses cumulative sums to compute the rolling standard deviation over a specified window size.
1. Unpack the shape of the input tensor `x` of shape (N,T,C), where N is the number of samples, T is the number of timesteps, and C is the number of channels (features).
2. We convert the input tensor to `float64` for better precision during cumulative sum calculations. Because rolling statistics can accumulate small numerical errors, using higher precision helps mitigate this.
3. We compute the cumulative sum `S` of the input tensor `x` along the time axis (`axis=1`). This gives us a tensor where each element at time `t` contains the sum of all previous values up to `t`. For each channel `c`, this can be expressed mathematically as:
   $$
   S(n, t, c) = \sum_{i=0}^{t} x(n, i, c)
   $$
  For example, if we have a time series $[x(0), x(1), x(2), ...]$, the cumulative sum at time $t=2$ would be $S(2) = x(0) + x(1) + x(2)$, and so on for each time step.
4. We compute the cumulative sum `S2` of the squared input tensor $x^2$ along the time axis. This gives us a tensor where each element at time `t` contains the sum of squares of all previous values up to $t$. Mathematically:
   $$
   S2(n, t, c) = \sum_{i=0}^{t} x(n, i, c)^2
   $$
  This is useful for calculating variance later on. For example, at time $t=2$, $S2(2) = x(0)^2 + x(1)^2 + x(2)^2$, and so on.
5. We initialize an output tensor `std` to hold the rolling standard deviation values.
6. We iterate over each time step `t` from 0 to T-1 to compute the rolling standard deviation for each time step:
   1. We determine the effective window size `w` for the current time step, which is the minimum of the specified window size and `t+1`. This ensures that at the beginning of the time series, we only consider available data points.
      $$
        w = \min(\text{window}, t+1)
      $$
   2. We compute the sum `s` of the values in the current window using the cumulative sum `S`:
      $$
      s(n, t, c) = S(n, t, c) - \alpha \quad \alpha = \begin{cases} S(n, t-w, c) & t-w \geq 0 \\ 0 & \text{otherwise} \end{cases}
      $$
      If $t-w < 0$, we simply take $S(n, t, c)$.
   3. We compute the sum of squares `s2` in the current window using the cumulative sum `S2`:
      $$
      s2(n, t, c) = S2(n, t, c) - \beta \quad \beta = \begin{cases} S2(n, t-w, c) & t-w \geq 0 \\ 0 & \text{otherwise} \end{cases}
      $$
      If $t-w < 0$, we simply take $S2(n, t, c)$.
   4. We calculate the mean of the values in the window:
      $$
      \text{mean}(n, t, c) = \mathbb{E}[s(n, t, c)] = \frac{s(n, t, c)}{w}
      $$
   5. We calculate the variance using the formula:
      $$
      \text{var}(n, t, c) = \frac{s2(n, t, c)}{w} - \text{mean}(n, t, c)^2
      $$
      We ensure that the variance is non-negative by taking the maximum with 0.0 (`np.maximum(var, 0.0)`).
   6. We compute the standard deviation by taking the square root of the variance, adding a small epsilon value to avoid taking the square root of zero:
      $$
      \text{std}(n, t, c) = \sqrt{\text{var}(n, t, c) + \varepsilon} \qquad \varepsilon = 1\mathrm{e}{-8}
      $$
7. Finally, we convert the output tensor `std` back to float32 and return it.

In [1]:
import numpy as np

def compute_delta(x: np.ndarray) -> np.ndarray:
    """
    Compute temporal derivative (delta) along time axis.
    :param x: (N,T,C) input tensor
    :return: (N,T,C) delta tensor
    """
    d = np.empty_like(x, dtype=np.float32)
    d[:,0,:] = 0.0
    d[:,1:,:] = x[:,1:,:] - x[:,:-1,:]
    return d

def rolling_std(x: np.ndarray, window=5, eps=1e-8) -> np.ndarray:
    """
    Compute rolling standard deviation over time axis.
    :param x: (N,T,C) input tensor
    :param window: rolling window size
    :param eps: small value to avoid sqrt(0)
    :return: (N,T,C) rolling std tensor
    """
    # unpack shape
    N,T,C = x.shape
    # we choose float64 instead of float32 for better precision in cumsum
    x = x.astype(np.float64)
    # cumulative sums over T
    # 1. S is cumulative sum of x
    S  = np.cumsum(x, axis=1)   # (N,T,C)
    # 2. S2 is cumulative sum of x^2
    S2 = np.cumsum(x*x, axis=1) # (N,T,C)
    # compute rolling std;
    # initialize output standard deviation tensor
    std = np.empty((N,T,C), dtype=np.float64)
    # for each time step of the original sequence
    for t in range(T):
        w = min(window, t+1)
        s  = S[:,t,:]  - (S[:,t-w,:]  if t-w >= 0 else 0)
        s2 = S2[:,t,:] - (S2[:,t-w,:] if t-w >= 0 else 0)
        mean = s / w
        var  = s2 / w - mean*mean
        var  = np.maximum(var, 0.0)
        std[:,t,:] = np.sqrt(var + eps)
    return std.astype(np.float32)

### 🏴‍☠️ Apply Feature Engineering to Pirate Pain Dataset

Here, we load the v1 artifact containing the original dataset, extract the dynamic features, and apply our feature engineering functions to compute the delta and rolling standard deviation for the numeric dynamic features. Finally, we save the enhanced dataset as a new v2 artifact.

In [2]:
from internal.data_types import DataSet, DataSetV2
from internal.utils import PersistenceManager

# load v1 artifact
data: DataSet = PersistenceManager.load_arrays_and_scalers()

# extract data
X_dyn_train = data.X_dyn_train  # (N,160,D_dyn)
X_dyn_val   = data.X_dyn_val
X_dyn_test  = data.X_dyn_test

# set the window for rolling std
ROLLING_STD_WINDOW = 5

def split_numeric_surveys(x_dyn: np.ndarray):
    """
    Splits dynamic tensor into numeric and survey parts.
    Assumes first 4 channels are surveys.
    :param x_dyn: (N,T,34) dynamic tensor
    :return:
    """
    return (
        # first 4 channels are integer (surveys)
        [x_dyn[:, :, 0 + i].astype(np.int64) for i in range(4)],
        # remaining channels are float (numeric)
        x_dyn[:, :, 4:].astype(np.float32)
    )

surv_tr, X_num_tr = split_numeric_surveys(X_dyn_train)
surv_te, X_num_te = split_numeric_surveys(X_dyn_test)
surv_va, X_num_va = split_numeric_surveys(X_dyn_val)

def build_engineered(x_num: np.ndarray, window=5) -> np.ndarray:
    """
    Build engineered dynamic numeric features with delta and rolling std.
    :param x_num: (N,T,30) input numeric dynamic features
    :param window: rolling std window size
    :return: (N,T,90) engineered dynamic numeric features
    """
    raw   = x_num.astype(np.float32)
    return np.concatenate([raw, compute_delta(raw), rolling_std(raw, window=window)], axis=-1)  # (N,T,90)

X_dyn_num_train = build_engineered(X_num_tr, ROLLING_STD_WINDOW)
X_dyn_num_test  = build_engineered(X_num_te, ROLLING_STD_WINDOW)
X_dyn_num_val = build_engineered(X_num_va, ROLLING_STD_WINDOW)

# Sanity checks
assert X_dyn_num_train.shape[1] == 160 and X_dyn_num_train.shape[2] == 90
assert X_dyn_num_test.shape[1]  == 160 and X_dyn_num_test.shape[2]  == 90
assert X_dyn_num_val.shape[1] == 160 and X_dyn_num_val.shape[2] == 90

# Update data and save v2 artifact
data.update({
    'X_dyn_num_train': X_dyn_num_train,   # (N,160,90)  numeric-only dynamic (engineered)
    'X_dyn_num_val':   X_dyn_num_val,     # None or (N,160,90)
    'X_dyn_num_test':  X_dyn_num_test,    # (1324,160,90)
    'feat_eng':        {'delta': True, 'rolling_std': True, 'window': 5},
})
PersistenceManager.save_arrays_v2(data, 0) # no compression for faster saves
print(
    f"Train dyn num: {X_dyn_num_train.shape}\n"
    f"Val dyn num: {X_dyn_num_val.shape}\n"
    f"Test dyn num: {X_dyn_num_test.shape}"
)

Arrays and scalers loaded successfully from: /home/andre/university/AN2DL-Challenge-1/notebooks/processed/arrays_and_scalers.joblib
Arrays v2 saved successfully to: /home/andre/university/AN2DL-Challenge-1/notebooks/processed/arrays_v2.joblib
Train dyn num: (528, 160, 90)
Val dyn num: (133, 160, 90)
Test dyn num: (1324, 160, 90)
